# Optimized Retail Cost Prediction Using Feature Engineering and Ensemble Learning

**Target:** `cost`  
**Model:** RandomForest (baseline) · GradientBoosting · HistGradientBoosting · VotingEnsemble  
**Workflow:** Data Loading → EDA → Preprocessing → Training → Evaluation → Interpretation

## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.ensemble import (HistGradientBoostingRegressor, RandomForestRegressor,
                               GradientBoostingRegressor, VotingRegressor)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 4)

## 2. Data Loading

In [ ]:
# Load datasets — update paths if needed
train_df = pd.read_csv('Dataset/train.csv')
test_df  = pd.read_csv('Dataset/test.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
train_df.head()

## 3. Data Inspection

In [ ]:
train_df.info()

In [ ]:
train_df.describe().T

In [ ]:
# Missing values check
missing = train_df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values')

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution — check if log transform is needed
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(train_df['cost'], bins=40, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Target Distribution: cost')

sns.boxplot(x=train_df['cost'], ax=axes[1], color='lightcoral')
axes[1].set_title('Boxplot: cost')

plt.tight_layout()
plt.show()

print(f'Skewness: {train_df["cost"].skew():.4f}')
# Skewness near 0 => no log transform needed on target

In [ ]:
# Skewness of all numeric features — only transform if |skew| > 1
skewness = train_df.drop(columns=['id', 'cost']).skew().sort_values(ascending=False)
print('Feature skewness:')
print(skewness.round(3))

In [ ]:
# Correlation with target — informs feature relevance
corr = train_df.drop(columns=['id']).corr()['cost'].drop('cost').sort_values(ascending=False)

plt.figure(figsize=(10, 4))
corr.plot(kind='bar', color='steelblue')
plt.title('Feature Correlation with cost')
plt.ylabel('Pearson r')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

# Note: low Pearson r doesn't mean a feature is useless —
# tree-based models capture non-linear relationships that correlation misses.

In [ ]:
# Distribution of key continuous features
cont_features = ['store_sales(in millions)', 'gross_weight', 'store_sqft', 'units_per_case']

fig, axes = plt.subplots(1, len(cont_features), figsize=(16, 4))
for ax, col in zip(axes, cont_features):
    sns.histplot(train_df[col], bins=30, kde=True, ax=ax, color='teal')
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Binary feature breakdown vs cost
binary_cols = ['coffee_bar', 'video_store', 'salad_bar', 'prepared_food', 'florist',
               'recyclable_package', 'low_fat']

fig, axes = plt.subplots(1, len(binary_cols), figsize=(20, 4))
for ax, col in zip(axes, binary_cols):
    train_df.groupby(col)['cost'].mean().plot(kind='bar', ax=ax, color=['#4C72B0', '#DD8452'])
    ax.set_title(f'Avg cost by {col}')
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

## 5. Preprocessing & Feature Engineering

In [ ]:
# Drop id — not a predictive feature
FEATURES = [c for c in train_df.columns if c not in ['id', 'cost']]
TARGET   = 'cost'

X = train_df[FEATURES].copy()
y = train_df[TARGET].copy()

X_holdout = test_df[FEATURES].copy()

print(f'Features used ({len(FEATURES)}): {FEATURES}')

In [ ]:
# Selective log transform — only for features with |skew| > 1
# num_children_at_home has skew ~1.85, so we apply log1p
# All other features have |skew| < 1 — no transform needed

SKEWED_COLS = ['num_children_at_home']  # identified from skewness analysis above

def apply_log_transform(df, cols):
    df = df.copy()
    for col in cols:
        df[col] = np.log1p(df[col])
    return df

X         = apply_log_transform(X, SKEWED_COLS)
X_holdout = apply_log_transform(X_holdout, SKEWED_COLS)

print('Log transform applied to:', SKEWED_COLS)

## 6. Train / Validation / Test Split

In [ ]:
# 70% train | 15% validation | 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED)

print(f'Train:      {X_train.shape[0]:>7,} rows')
print(f'Validation: {X_val.shape[0]:>7,} rows')
print(f'Test:       {X_test.shape[0]:>7,} rows')

## 7. Model Training

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_ev, y_ev):
    """Fit model and print R2, MAE, RMSE on train and eval sets."""
    model.fit(X_tr, y_tr)
    for split_name, Xs, ys in [('Train', X_tr, y_tr), ('Eval', X_ev, y_ev)]:
        preds = model.predict(Xs)
        r2   = r2_score(ys, preds)
        mae  = mean_absolute_error(ys, preds)
        rmse = np.sqrt(mean_squared_error(ys, preds))
        print(f'{name} [{split_name}]  R2={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}')
    return model

In [ ]:
# --- Baseline: Random Forest ---
rf = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)
rf = evaluate('RandomForest', rf, X_train, y_train, X_val, y_val)

In [ ]:
# --- Primary: HistGradientBoostingRegressor ---
# Faster than GradientBoostingRegressor on large datasets,
# natively handles missing values, and generally outperforms RF.
hgb = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_depth=6,
    min_samples_leaf=20,
    random_state=SEED
)
hgb = evaluate('HistGradBoost', hgb, X_train, y_train, X_val, y_val)

In [ ]:
# --- Boosting: GradientBoostingRegressor ---
# Classic sklearn boosting — builds trees sequentially, each correcting
# the residuals of the previous. Slower than HistGBM but useful for comparison.
gbr = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    min_samples_leaf=20,
    subsample=0.8,
    random_state=SEED
)
gbr = evaluate('GradientBoost', gbr, X_train, y_train, X_val, y_val)

In [ ]:
# --- Ensemble: VotingRegressor ---
# Combines RF, GBR, and HistGBM by averaging their predictions.
# Reduces variance and often improves generalization over any single model.
ensemble = VotingRegressor(estimators=[
    ('rf',  RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)),
    ('gbr', GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                      max_depth=5, subsample=0.8, random_state=SEED)),
    ('hgb', HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05,
                                          max_depth=6, min_samples_leaf=20, random_state=SEED))
], n_jobs=-1)
ensemble = evaluate('VotingEnsemble', ensemble, X_train, y_train, X_val, y_val)

### Optional: Hyperparameter Tuning

In [ ]:
# Uncomment to run — takes a few minutes on large data

# param_dist = {
#     'max_iter':       [200, 300, 500],
#     'learning_rate':  [0.01, 0.05, 0.1],
#     'max_depth':      [4, 6, 8],
#     'min_samples_leaf': [10, 20, 50],
# }
# search = RandomizedSearchCV(
#     HistGradientBoostingRegressor(random_state=SEED),
#     param_distributions=param_dist,
#     n_iter=20, cv=3, scoring='r2',
#     random_state=SEED, n_jobs=-1, verbose=1
# )
# search.fit(X_train, y_train)
# print('Best params:', search.best_params_)
# hgb = search.best_estimator_

## 8. Final Evaluation on Test Set

In [ ]:
# Evaluate all models on held-out test set and compare
results = []
for name, model in [('RandomForest', rf), ('GradientBoost', gbr),
                     ('HistGradBoost', hgb), ('VotingEnsemble', ensemble)]:
    preds = model.predict(X_test)
    results.append({
        'Model': name,
        'R²':   round(r2_score(y_test, preds), 4),
        'MAE':  round(mean_absolute_error(y_test, preds), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, preds)), 4)
    })

results_df = pd.DataFrame(results).sort_values('R²', ascending=False)
print('=== Test Set Comparison ===')
print(results_df.to_string(index=False))

# Use best model for plots and submission
best_name  = results_df.iloc[0]['Model']
best_model = dict(zip(
    ['RandomForest', 'GradientBoost', 'HistGradBoost', 'VotingEnsemble'],
    [rf, gbr, hgb, ensemble]
))[best_name]
y_pred_test = best_model.predict(X_test)
print(f'\nBest model: {best_name}')

In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred_test, alpha=0.3, s=5, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', linewidth=1.5, label='Perfect prediction')
plt.xlabel('Actual cost')
plt.ylabel('Predicted cost')
plt.title('Actual vs Predicted — Best Model')
plt.legend()
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Residual plot — check for systematic bias
residuals = y_test - y_pred_test

plt.figure(figsize=(8, 4))
plt.scatter(y_pred_test, residuals, alpha=0.3, s=5, color='coral')
plt.axhline(0, color='black', linewidth=1)
plt.xlabel('Predicted cost')
plt.ylabel('Residual')
plt.title('Residual Plot')
plt.tight_layout()
plt.show()

print(f'Residual mean: {residuals.mean():.4f}  (should be near 0)')

## 9. Feature Importance & Interpretation

In [ ]:
# Permutation importance — model-agnostic, more reliable than impurity-based importance
perm = permutation_importance(best_model, X_test, y_test, n_repeats=10,
                               random_state=SEED, n_jobs=-1)

imp_df = pd.DataFrame({
    'Feature':    FEATURES,
    'Importance': perm.importances_mean,
    'Std':        perm.importances_std
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=imp_df, x='Importance', y='Feature', palette='Blues_r')
plt.title('Permutation Feature Importance (Test Set)')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(imp_df.to_string(index=False))

## 10. Generate Submission Predictions

In [ ]:
# Predict on the holdout test set and save submission
holdout_preds = best_model.predict(X_holdout)

submission = pd.DataFrame({
    'id':   test_df['id'],
    'cost': holdout_preds
})

submission.to_csv('Dataset/sample_submission.csv', index=False)
print('Submission saved.')
submission.head()

## 11. Conclusion

**What was done:**
- Loaded and inspected the dataset (360k train rows, 15 features, no missing values)
- EDA confirmed the target `cost` is nearly symmetric — no log transform needed on target
- Applied selective log transform only to `num_children_at_home` (skew ~1.85)
- Trained a RandomForestRegressor as baseline, GradientBoostingRegressor, HistGradientBoostingRegressor, and a VotingEnsemble combining all three
- Evaluated with R², MAE, RMSE on a held-out test split
- Used permutation importance for reliable feature interpretation

**Why HistGradientBoostingRegressor?**
- Handles large tabular datasets efficiently (histogram-based binning)
- Captures non-linear relationships that Lasso/Linear models miss
- Natively handles missing values (no imputation needed)
- Consistently outperforms Random Forest with less memory usage
- Faster training than standard GradientBoostingRegressor

**Optional next improvements:**
1. Run `RandomizedSearchCV` (cell above) for hyperparameter tuning
2. Try `LightGBM` or `XGBoost` — often faster and slightly better on tabular data
3. Add k-fold cross-validation for more robust R² estimates
4. Engineer interaction features (e.g., `store_sqft * store_sales`)
5. Use SHAP values for deeper model explainability